# TextMamba3D — A100 Training Pipeline (V5.0)

**V5.0: Mamba-3 Complex-Valued SSM Backbone**

| Feature | Description |
|---------|-------------|
| SSM Backend | Mamba-3 complex-valued SSM with RoPE + trapezoidal discretization |
| d_state | 16 (same as V4.5, isolates Mamba-3 effect) |
| headdim | 48 (divides all stage dims: 48, 96, 192, 384) |
| A100 40GB | batch_size=2, gradient_accumulation=2 (effective batch=4) |
| Checkpoint | **Incompatible** with Mamba-1 — must train from scratch |

Config: `configs/textbrats_a100_v5.yaml`

> **Key change from V4.6:** swaps the inner SSM from Mamba-1 to Mamba-3 across
> the entire image encoder/decoder pipeline. The text encoder (lightweight
> MambaLayer adapter over frozen PubMedBERT) stays on Mamba-1.
> Targets TC Dice regression via complex-valued rotational dynamics across
> 3-axis cross-scan (DHW, HWD, WDH).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# GPU check
!nvidia-smi 2>/dev/null || echo "No GPU detected (CPU mode)"

# Install dependencies
# IMPORTANT: mamba-ssm must be installed from GitHub source to get Mamba3 support
# PyPI mamba-ssm (<=2.2.x) does NOT include Mamba3
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    causal-conv1d \
    transformers nibabel tensorboard pyyaml tqdm

# Install mamba-ssm from source (includes Mamba3)
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    git+https://github.com/state-spaces/mamba.git@v2.2.4

# Verify Mamba3 is importable
import importlib
mamba_ssm = importlib.import_module('mamba_ssm')
assert hasattr(mamba_ssm, 'Mamba3'), "Mamba3 not found in mamba_ssm! Source install may have failed."
print(f"mamba_ssm version: {mamba_ssm.__version__}")
print("Mamba3 import: OK")

In [ ]:
import os, zipfile, shutil, subprocess, time

REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'

# Code source: git clone (public repo)
# If old zip-extracted dir exists (no .git), remove and re-clone
git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    print(f'Removing old non-git code at {REPO_DIR}...')
    shutil.rmtree(REPO_DIR)

if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
    print(f'Updated existing repo at {REPO_DIR}')
else:
    # Clone with retry
    for attempt in range(1, 4):
        print(f'Cloning (attempt {attempt}/3)...')
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True
        )
        if ret.returncode == 0 and os.path.exists(os.path.join(REPO_DIR, 'models/textmamba3d.py')):
            break
        print(f'  Failed (code {ret.returncode}): {ret.stderr.strip()}')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        if attempt < 3:
            time.sleep(5 * attempt)
    else:
        raise RuntimeError(f'Clone failed after 3 attempts. Last error: {ret.stderr.strip()}')
    os.chdir(REPO_DIR)
    print(f'Cloned to {REPO_DIR}')

print(f'Working directory: {os.getcwd()}')

# Extract BraTS data from Drive
DATA_ZIP = os.path.join(DRIVE_BASE, "TextBraTS_data.zip")
DATA_DIR = os.path.join(REPO_DIR, "data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData")

if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    if os.path.exists(DATA_ZIP):
        print(f"Extracting {DATA_ZIP}...")
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(os.path.dirname(DATA_DIR))
        if os.path.exists(DATA_DIR):
            print(f"Data extracted. Cases: {len(os.listdir(DATA_DIR))}")
        else:
            print(f"ERROR: Expected path not found after extraction: {DATA_DIR}")
            print("Actual contents:", os.listdir(os.path.dirname(DATA_DIR)))
    else:
        print(f"ERROR: {DATA_ZIP} not found on Drive")
else:
    print(f"Data already exists. Cases: {len(os.listdir(DATA_DIR))}")

# Count samples
if os.path.exists(DATA_DIR):
    cases = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"Total BraTS cases: {len(cases)}")

In [ ]:
import sys, yaml, inspect
sys.path.insert(0, '.')

# Verify V5.0 Mamba3 modules
from models.mamba_block import (
    _create_ssm, _auto_headdim, MAMBA3_AVAILABLE,
    MambaBlock, BiMambaBlock, CrossScanBiMamba3DBlock,
)
print(f"MAMBA3_AVAILABLE: {MAMBA3_AVAILABLE}")
assert MAMBA3_AVAILABLE, "Mamba3 not available! Check mamba-ssm installation."

# Verify _auto_headdim for V5.0 stage dimensions
for dim in [48, 96, 192, 384]:
    d_inner = dim * 2  # expand=2
    hd = _auto_headdim(d_inner)
    assert d_inner % hd == 0, f"headdim={hd} invalid for d_inner={d_inner}"
    print(f"  Stage dim={dim} -> d_inner={d_inner} -> auto headdim={hd}")

# Verify TextMamba3D has V5.0 params
from models.textmamba3d import TextMamba3D
sig = inspect.signature(TextMamba3D.__init__)
for param in ['use_mamba3', 'headdim']:
    assert param in sig.parameters, f"TextMamba3D missing V5.0 param: {param}!"
print(f"TextMamba3D: use_mamba3, headdim params present")

# Verify V4.6 params still present (backward compat)
for param in ['use_text_gate', 'use_cross_scale_skip', 'text_gate_init_bias']:
    assert param in sig.parameters, f"TextMamba3D missing V4.6 param: {param}!"
print(f"TextMamba3D: V4.6 backward compat OK")

# Verify config
with open('configs/textbrats_a100_v5.yaml') as f:
    cfg = yaml.safe_load(f)
assert cfg['model']['use_mamba3'] is True, "use_mamba3 should be True"
assert cfg['model']['headdim'] == 48, f"headdim should be 48, got {cfg['model']['headdim']}"
assert cfg['model']['d_state'] == 64, f"d_state should be 64, got {cfg['model']['d_state']}"
assert cfg['data']['batch_size'] == 2, f"batch_size should be 2, got {cfg['data']['batch_size']}"
assert cfg['training']['gradient_accumulation'] == 2, "gradient_accumulation should be 2"
assert cfg['training']['gradient_checkpointing'] is True, "gradient_checkpointing should be True"
assert cfg['data'].get('et_enriched') is True, "et_enriched should be True"
print(f"Config: V5.0 Mamba3 settings verified")
print(f"  use_mamba3={cfg['model']['use_mamba3']}, headdim={cfg['model']['headdim']}, d_state={cfg['model']['d_state']}")
print(f"  batch_size={cfg['data']['batch_size']}, gradient_accumulation={cfg['training']['gradient_accumulation']}")
print(f"  effective_batch_size={cfg['data']['batch_size'] * cfg['training']['gradient_accumulation']}")
print(f"  et_enriched={cfg['data']['et_enriched']}, enriched_prob={cfg['data'].get('enriched_prob', 'N/A')}")

# Mamba3 SSM smoke test (requires CUDA)
import torch
assert torch.cuda.is_available(), (
    "This notebook requires a CUDA GPU runtime. "
    "Go to Runtime -> Change runtime type -> GPU (A100)."
)
from mamba_ssm import Mamba3
device = torch.device('cuda')
ssm = Mamba3(d_model=48, d_state=64, expand=2, headdim=48).to(device)
x = torch.randn(1, 64, 48).to(device)
out = ssm(x)
assert out.shape == x.shape, f"Shape mismatch: {out.shape} != {x.shape}"
print(f"Mamba3 smoke test: input {x.shape} -> output {out.shape} OK")
del ssm, x, out
torch.cuda.empty_cache()

print()
print("All V5.0 modules verified!")

In [ ]:
import os, sys, zipfile
os.chdir(REPO_DIR)

DATA_DIR = "./data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
ET_CACHE_ZIP = os.path.join(DRIVE_BASE, "et_enriched.zip")

# Safety check: verify dataset exists and is non-empty
cases = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
) if os.path.isdir(DATA_DIR) else []
if not cases:
    raise RuntimeError(
        f"No BraTS cases found in {DATA_DIR}. "
        "Check that data extraction completed successfully."
    )

sample_case = cases[0]
sample_enriched = os.path.join(DATA_DIR, sample_case, f"{sample_case}_et_enriched.txt")

if os.path.exists(sample_enriched):
    # Already generated (same runtime)
    count = sum(
        1 for d in cases
        if os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt"))
    )
    print(f"ET-enriched text already present for {count} cases, skipping")
    for case in cases[:3]:
        path = os.path.join(DATA_DIR, case, f"{case}_et_enriched.txt")
        if os.path.exists(path):
            with open(path, 'r') as f:
                print(f"  {case}: {f.read().strip()[:120]}...")

elif os.path.exists(ET_CACHE_ZIP):
    # Restore from Drive cache
    print(f"Restoring ET-enriched text from {ET_CACHE_ZIP}...")
    with zipfile.ZipFile(ET_CACHE_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    count = sum(
        1 for d in cases
        if os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt"))
    )
    print(f"Restored ET-enriched text for {count} cases from Drive cache")

else:
    # Generate + cache to Drive
    print("Generating ET-enriched text descriptions from T1ce images...")
    sys.path.insert(0, '.')
    from data.et_text_enrichment import process_all_cases
    results = process_all_cases(DATA_DIR)

    no_enhancement = sum(1 for desc in results.values() if "No significant" in desc)
    total = len(results)
    print(f"Total: {total}, No enhancement: {no_enhancement} ({no_enhancement/total*100:.1f}%)")

    # Cache to Drive for next runtime
    with zipfile.ZipFile(ET_CACHE_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
        for case_dir in cases:
            et_file = os.path.join(DATA_DIR, case_dir, f"{case_dir}_et_enriched.txt")
            if os.path.exists(et_file):
                zf.write(et_file, os.path.join(case_dir, f"{case_dir}_et_enriched.txt"))
    print(f"Cached ET text to {ET_CACHE_ZIP}")

# Hard check: fail fast if ET-enriched text is missing
et_count = sum(
    1 for d in cases
    if os.path.exists(os.path.join(DATA_DIR, d, f'{d}_et_enriched.txt'))
)
if et_count == 0:
    raise RuntimeError(
        f'et_enriched=true in config but 0 ET-enriched text files found in {DATA_DIR}. '
        'Run the ET text generation step first (see V4.5 training notebook Cell 4).'
    )
print(f'ET-enriched text verified: {et_count}/{len(cases)} cases')


In [ ]:
# Smoke test: 2 samples, 1 epoch — verify no OOM or shape mismatch
import subprocess
os.chdir(REPO_DIR)

print("Running smoke test (2 samples, 1 epoch)...")
ret = subprocess.run([
    "python", "-u", "train.py",
    "--config", "configs/textbrats_a100_v5.yaml",
    "--max-samples", "2",
    "--max-epochs", "1",
    "--no-text-ratio", "0.0",
    "--grad-accum", "1",
])
if ret.returncode != 0:
    raise RuntimeError("Smoke test FAILED. Fix errors before full training.")
print("Smoke test PASSED")

# Check GPU memory after smoke
import torch
if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"Peak GPU memory: {peak:.1f} / {total:.0f} GB")
    torch.cuda.reset_peak_memory_stats()


## Training (A100 40GB)

| Parameter | Value | Note |
|-----------|-------|------|
| batch_size | 2 | Reduced from 4 (Mamba3 d_state=64 uses more memory) |
| gradient_accumulation | 2 | Maintains effective batch size of 4 |
| gradient_checkpointing | true | Required for A100 40GB |
| sw_batch_size | 2 | Sliding window inference |
| num_workers | 4 | Colab A100 |
| d_state | 16 | Same as V4.5 (isolate Mamba-3 effect) |
| headdim | 48 | Divides all stage dims |

> **Warning:** Mamba-3 weights are architecturally incompatible with Mamba-1.
> V5.0 must train from scratch — do NOT resume from V4.x checkpoints.

In [ ]:
import os, shutil, glob

DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
os.makedirs(DRIVE_CKPT, exist_ok=True)

def sync_checkpoints_to_drive():
    local_ckpt = os.path.join(REPO_DIR, "checkpoints")
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, "*.pth")):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(f))
        shutil.copy2(f, dst)
    print(f"Synced checkpoints to {DRIVE_CKPT}")

# Clean local checkpoints (V5.0 trains from scratch)
for f in glob.glob(os.path.join(REPO_DIR, "checkpoints/*.pth")):
    os.remove(f)
print("Cleaned local checkpoints for V5.0 fresh start")
print("(V4.x checkpoints preserved on Drive but incompatible with V5.0)")

In [ ]:
import subprocess, shutil
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

# V5.0 training: Mamba-3 SSM backbone + ET-Enriched (A100 40GB)
# batch=2, grad-accum=2 -> effective batch=4
ret = subprocess.run(
    ['python', '-u', 'train.py',
     '--config', 'configs/textbrats_a100_v5.yaml',
     '--no-text-ratio', '0.15',
     '--grad-accum', '2'],
    cwd=REPO_DIR,
)
if ret.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {ret.returncode}. Check output above.")

# Verify checkpoint was produced before syncing
local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
local_last = os.path.join(REPO_DIR, "checkpoints/last.pth")
if not os.path.exists(local_best) and not os.path.exists(local_last):
    raise RuntimeError("Training completed but no checkpoint was saved. Check train.py output.")

# Sync and save
sync_checkpoints_to_drive()

best_ckpt = os.path.join(DRIVE_CKPT, "best_v5.0.pth")
if os.path.exists(local_best):
    shutil.copy2(local_best, best_ckpt)
    print(f"Best checkpoint saved: {best_ckpt}")

## Evaluation

In [ ]:
os.chdir(REPO_DIR)

ckpt = os.path.join(REPO_DIR, "checkpoints/best.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, "best_v5.0.pth")

if os.path.exists(ckpt):
    print("=" * 60)
    print("Evaluation: With Text (Mamba3 + SeqCA)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_a100_v5.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --use-text \
        --overlap 0.5

    print()

    print("=" * 60)
    print("Evaluation: Without Text (fusion bypassed)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_a100_v5.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --no-text \
        --overlap 0.5

    print()
    print("=" * 60)
    print("Compare: with-text Dice - without-text Dice = text guidance delta")
    print("V4.6 baseline: ~88% Mean Dice")
    print("V5.0 target: >= 88% with improved TC class")
    print("Primary metric: TC Dice improvement (complex SSM -> better rotational dynamics)")
    print("=" * 60)
else:
    print(f"No checkpoint found at {ckpt}")
    print("Run training first")

In [ ]:
import matplotlib.pyplot as plt

# Fill in actual results after evaluation
v46_dice = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 88.0}  # V4.6 baseline
v50_dice = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 0.0}   # Fill after eval

if v50_dice['Mean'] == 0.0:
    print("V5.0 results not yet filled in.")
    print("Update v46_dice and v50_dice dictionaries after evaluation, then re-run this cell.")
    print()
    print("Key metrics to watch:")
    print("  - TC Dice: primary success metric (complex SSM targets rotational dynamics)")
    print("  - Mean Dice: should be >= V4.6 baseline (~88%)")
    print("  - ET Dice: should not regress")
else:
    labels = list(v46_dice.keys())
    v46_vals = list(v46_dice.values())
    v50_vals = list(v50_dice.values())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart comparison
    x = range(len(labels))
    w = 0.35
    ax1.bar([i - w/2 for i in x], v46_vals, w, label='V4.6 (Mamba-1)', color='steelblue', alpha=0.8)
    ax1.bar([i + w/2 for i in x], v50_vals, w, label='V5.0 (Mamba-3)', color='coral', alpha=0.8)
    ax1.set_ylabel('Dice (%)')
    ax1.set_title('V4.6 vs V5.0 Dice Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    # Delta chart
    deltas = [v50 - v46 for v46, v50 in zip(v46_vals, v50_vals)]
    colors = ['green' if d >= 0 else 'red' for d in deltas]
    ax2.bar(labels, deltas, color=colors, alpha=0.8)
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_ylabel('Delta (%)')
    ax2.set_title('V5.0 - V4.6 Improvement')
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('v50_a100_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: v50_a100_comparison.png")

## Resume Training (After Disconnect)

> **Important:** Only resume from V5.0 checkpoints. V4.x checkpoints are incompatible.

In [ ]:
import os, shutil
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

resume_ckpt = os.path.join(DRIVE_CKPT, "last.pth")
if os.path.exists(resume_ckpt):
    print(f"Resuming from {resume_ckpt}")
    !python train.py \
        --config configs/textbrats_a100_v5.yaml \
        --resume "{resume_ckpt}" \
        --no-text-ratio 0.15 \
        --grad-accum 2

    sync_checkpoints_to_drive()

    best_ckpt = os.path.join(DRIVE_CKPT, "best_v5.0.pth")
    local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
    if os.path.exists(local_best):
        shutil.copy2(local_best, best_ckpt)
        print(f"Best checkpoint saved: {best_ckpt}")
else:
    print("No checkpoint to resume from.")
    print(f"Expected: {resume_ckpt}")
    print("Run training first")